## Setup

In [1]:
#Enables module automatic reload. 
#Your notebook will be able to pick up code updates made to qiskit-metal (or other) module code.
%reload_ext autoreload
%autoreload 2

In [2]:
from qiskit_metal import designs, MetalGUI, Dict, Headings

In [3]:
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond

#Explore the options of the LaunchpadWirebond
#LaunchpadWirebond.get_template_options(design)

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight

#Explore the options of the RouteMeander.
#RouteMeander.get_template_options(design)

from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
#CoupledLineTee.get_template_options(design)

from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

#Explore the options of the ShortToGround.
#ShortToGround.get_template_options(design)

from qiskit_metal.qlibrary.user_components import ClawCoupler

#Explore the options of the RouteMeander.
#ClawCoupler.get_template_options(design)

## Create chip

In [4]:
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '5mm'
design.chips.main.size['size_y'] = '5mm'

gui = MetalGUI(design)

# If you disable the next line with "overwrite_enabled", then you will need to 
# delete a component [<component>.delete()] before recreating it.
design.overwrite_enabled = True

07:12PM 27s CRITICAL [_qt_message_handler]: line: 0, func: None(), file: None  WARNING: Populating font family aliases took 123 ms. Replace uses of missing font family "Courier" with one that exists to avoid this cost. 



### Transmission line

Parameters

In [5]:
coupler_length = 100 # in unit of um

2025-02-22 19:12:28.282 python[88970:10887741] +[IMKClient subclass]: chose IMKClient_Modern
2025-02-22 19:12:28.282 python[88970:10887741] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [6]:
lp1 = LaunchpadWirebond(design, 'launch1', options=dict(pos_x='-1.5mm', pos_y='1mm', 
                                                        orientation='270',
                                                        pad_width='240um',
                                                        pad_height='160um',
                                                        pad_gap='174um'
                                                        ))
lp2 = LaunchpadWirebond(design, 'launch2', options=dict(pos_x='-1.5mm', pos_y='-1mm', 
                                                        orientation = '90',
                                                        pad_width='240um',
                                                        pad_height='160um',
                                                        pad_gap='174um'
                                                       ))
clt1 = CoupledLineTee(design, 'res_cpl1', options=dict(pos_x='-1.5mm', pos_y='{}um'.format(0 - coupler_length/2),
                                                       orientation='90',
                                                       coupling_length='{}um'.format(coupler_length),
                                                       open_termination=False
                                                       ))

In [7]:
scr1 = ClawCoupler(design, 'claw1', 
                 options=dict(pos_x='1mm', pos_y='0mm',
                              orientation='0',
                              #claw_cpw_width='2um',claw_gap='1um',
                              claw_spacing = '0.8um',claw_width='2um',
                              claw_length = '200um',
                              claw_tip_width='0um',claw_tip_spacing='4um'))

SCR_end = RouteMeander(design, 'res1', options=dict(
    total_length = '6mm',
    fillet = '99um',
    lead=Dict(start_straight='200um',
    end_straight='200um',
    #start_jogged_extension': '',
    #end_jogged_extension': ''},
             ),
    pin_inputs=Dict(
        end_pin=Dict(component=clt1.name, pin='second_end'),
        start_pin=Dict(component=scr1.name, pin='claw')
    )))

In [11]:
tline1 = RouteStraight(design, 'bus1', options=dict(
        pin_inputs=Dict(
        end_pin=Dict(component=lp1.name, pin='tie'),
        start_pin=Dict(component=clt1.name, pin='prime_end')
    )))
tline2 = RouteStraight(design, 'bus2', options=dict(
        pin_inputs=Dict(
        end_pin=Dict(component=clt1.name, pin='prime_start'),
        start_pin=Dict(component=lp2.name, pin='tie')
    )))

## Export to gds

In [9]:
gui.rebuild()
gui.autoscale()
a_gds = design.renderers.gds
a_gds.options
a_gds.options['no_cheese']['buffer'] = '50um'

In [10]:
a_gds.options['path_filename'] = 'Split_res.GDS'
a_gds.export_to_gds("Split_res.gds")

ValueError: [GDSPY] Multiple cells with name: TOP in GDSII file

## Generate gds with different claw spacing

In [13]:
for cs in [0.8,1.5,4,8,10,20]:
    scr1 = ClawCoupler(design, 'claw1', 
                     options=dict(pos_x='1mm', pos_y='0mm',
                                  orientation='0',
                                  #claw_cpw_width='2um',claw_gap='1um',
                                  claw_spacing = '{}um'.format(cs),claw_width='2um',
                                  claw_length = '200um',
                                  claw_tip_width='0um',claw_tip_spacing='4um'))
    gui.rebuild()
    gui.autoscale()
    a_gds = design.renderers.gds
    a_gds.options
    a_gds.options['no_cheese']['buffer'] = '50um'
    a_gds.options['path_filename'] = 'Split_res_{}um.GDS'.format(cs)
    a_gds.export_to_gds(a_gds.options['path_filename'])

07:17PM 45s WARNING [_import_junctions_to_one_cell]: Not able to find file:"Split_res_0.8um.GDS".  Not used to replace junction. Checked directory:"/Users/chenmo/Documents/GitHub/qiskit-metal/Quantum Acoustics".
07:17PM 45s WARNING [_import_junctions_to_one_cell]: Not able to find file:"Split_res_1.5um.GDS".  Not used to replace junction. Checked directory:"/Users/chenmo/Documents/GitHub/qiskit-metal/Quantum Acoustics".
07:17PM 46s WARNING [_import_junctions_to_one_cell]: Not able to find file:"Split_res_4um.GDS".  Not used to replace junction. Checked directory:"/Users/chenmo/Documents/GitHub/qiskit-metal/Quantum Acoustics".
07:17PM 46s WARNING [_import_junctions_to_one_cell]: Not able to find file:"Split_res_8um.GDS".  Not used to replace junction. Checked directory:"/Users/chenmo/Documents/GitHub/qiskit-metal/Quantum Acoustics".
07:17PM 46s WARNING [_import_junctions_to_one_cell]: Not able to find file:"Split_res_10um.GDS".  Not used to replace junction. Checked directory:"/Users/ch

## Getting info

In [ ]:
# Get a list of all the qcomponents in QDesign and then zoom on them.
all_component_names = design.components.keys()

gui.zoom_on_components(all_component_names)

In [ ]:
all_component_names

In [ ]:
CoupledLineTee.get_template_options(design)